# Main table

Reads one finished eval JSON and prints Table 1. No models are run.

`align = 1 - js_dist_gate`.

**VLM (VQAv2).** `SLoc` is `metrics.soft_generality`: the original image shifted toward each answer. `SLoc-OOD` is `metrics.blend_generality`: the same shift on a generated image. Both are the strength-weighted mean of alignment on the interpolated gold (`"0.1"` … `"1.0"`, not `"*_edit"`), weight = strength. `SGen-I` is the mean of image-generality alignment and `blend_generality` edit-gold alignment at `"0.0_edit"` and `"0.3_edit"`.

**LLM (ProtoQA).** `SGen` is `text_generality`. `SLoc` is `soft_generality` (answer-specific premises). There is no image set.

In [ ]:
import json
from pathlib import Path

# Saved editeval JSON from revlm.run.edit or rellm.run.edit.
PATH = ""  # e.g. results/softedit/.../postedit_qa_all.json


def align(block):
    if not isinstance(block, dict) or block.get("js_dist_gate") is None:
        return None
    return 1.0 - float(block["js_dist_gate"])


def acc(block):
    if not isinstance(block, dict) or block.get("simple_acc") is None:
        return None
    return float(block["simple_acc"])


def mean(vals):
    xs = [v for v in vals if v is not None]
    return sum(xs) / len(xs) if xs else None


def at(group, t):
    group = group or {}
    return group.get(f"{float(t):.1f}")


def weighted_align(group, strengths):
    """sum_t t * align(t) / sum_t t on the interpolated gold."""
    num = den = 0.0
    for t in strengths:
        a = align(at(group, t))
        if a is None:
            continue
        num += t * a
        den += t
    return num / den if den else None


def load(path):
    text = Path(path).read_text()
    if str(path).endswith(".jsonl"):
        rows = [json.loads(line) for line in text.splitlines() if line.strip()]
        return rows[-1]
    return json.loads(text)


def vlm_row(result):
    m = result["metrics"]
    rel = m.get("reliability") or {}
    tgen = m.get("text_generality") or {}
    igen = m.get("image_generality") or {}
    soft = m.get("soft_generality") or {}
    blend = m.get("blend_generality") or {}
    return {
        "Align": align(rel),
        "SGen-I": mean([align(igen), align(blend.get("0.0_edit")), align(blend.get("0.3_edit"))]),
        "SGen-T": align(tgen),
        "SLoc": weighted_align(soft, (0.1, 0.3, 0.5, 0.7, 1.0)),
        "SLoc-OOD": weighted_align(blend, (0.3, 0.5, 0.7, 1.0)),
        "Acc": acc(rel),
        "Gen-I": acc(igen),
        "Gen-T": acc(tgen),
        "Loc": result.get("locality"),
    }


def llm_row(result):
    m = result["metrics"]
    rel = m.get("reliability") or {}
    tgen = m.get("text_generality") or {}
    soft = m.get("soft_generality") or {}
    return {
        "Align": align(rel),
        "SGen": align(tgen),
        "SLoc": align(soft),
        "Acc": acc(rel),
        "Gen": acc(tgen),
        "Loc": result.get("locality"),
    }


def show(row):
    for k, v in row.items():
        print(f"{k:<10} {v:.3f}" if isinstance(v, float) else f"{k:<10} -")


if not PATH:
    print("Set PATH to a saved eval JSON.")
else:
    result = load(PATH)
    metrics = result.get("metrics") or {}
    if "blend_generality" in metrics or "image_generality" in metrics:
        print("VLM")
        show(vlm_row(result))
    else:
        print("LLM")
        show(llm_row(result))